0. 준비: 라이브러리 설치

In [ ]:
pip install cryptography

1. RSA 공개키/개인키 생성 코드

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

def generate_keys():
    # 2048비트 RSA 키 생성
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )

    public_key = private_key.public_key()

    # 개인키를 PEM 형식으로 저장 (비밀번호 없이 예시)
    with open("private_key.pem", "wb") as f:
        f.write(
            private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.NoEncryption(),
            )
        )

    # 공개키를 PEM 형식으로 저장
    with open("public_key.pem", "wb") as f:
        f.write(
            public_key.public_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PublicFormat.SubjectPublicKeyInfo,
            )
        )

    print("키 생성 완료: private_key.pem, public_key.pem")

키 생성 완료: private_key.pem, public_key.pem


2.1. 파일 전송

In [ ]:
# 서버가 암호화해서 전송할 파일 경로
path = "API_KEY.env"

In [ ]:
import socket
import struct
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
import os

# 전송할 파일 경로
path = "secret_data.txt"   # 예시

# 서버 실행
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind(("0.0.0.0", 5000))
server.listen(1)

print("서버 대기중...")

conn, addr = server.accept()
print(f"클라이언트 접속: {addr}")

# ================================
# 1) 클라이언트 공개키 수신
# ================================
key_len_bytes = conn.recv(4)
key_len = struct.unpack(">I", key_len_bytes)[0]
client_public_key_bytes = conn.recv(key_len)

client_public_key = serialization.load_pem_public_key(client_public_key_bytes)
print("클라이언트 공개키 수신 완료")

# ================================
# 2) 파일 읽기
# ================================
file_name = os.path.basename(path)
file_name_bytes = file_name.encode("utf-8")

with open(path, "rb") as f:
    file_data = f.read()

# ================================
# 3) 파일을 공개키로 암호화
# ================================
encrypted_data = client_public_key.encrypt(
    file_data,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

print(f"파일 암호화 완료 ({len(encrypted_data)} bytes)")

# ================================
# 4) 파일명 + 암호문 전송
# ================================

# (1) 파일명 길이 + 파일명
conn.sendall(struct.pack(">I", len(file_name_bytes)))
conn.sendall(file_name_bytes)

# (2) 암호문 길이 + 암호문
conn.sendall(struct.pack(">I", len(encrypted_data)))
conn.sendall(encrypted_data)

print("파일명 + 암호문 전송 완료")

conn.close()
server.close()


서버 대기중...
클라이언트 접속: ('192.168.75.62', 56370)
복호화된 데이터: hello secret key


2.2. 파일 수신

In [ ]:
SERVER_IP = ""

In [ ]:
import socket
import struct
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes

# ==========================
# 접속할 서버 IP 입력
# ==========================
SERVER_IP = ""   # ← 여기에 직접 입력하세요
PORT = 5000

# ==========================
# 1. 클라이언트 RSA 키 생성
# ==========================
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

# 공개키 PEM 직렬화
public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

public_key_len = len(public_pem)

# ==========================
# 2. 서버 접속
# ==========================
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect((SERVER_IP, PORT))
print("서버 접속 완료")

# ==========================
# 3. 클라이언트 공개키 전송
# ==========================
client.sendall(struct.pack(">I", public_key_len))
client.sendall(public_pem)
print("클라이언트 공개키 전송 완료")

# ==========================
# 4. 서버로부터 파일명 수신
# ==========================

# (1) 파일명 길이 수신
name_len_bytes = client.recv(4)
name_len = struct.unpack(">I", name_len_bytes)[0]

# (2) 파일명 자체 수신
file_name_bytes = client.recv(name_len)
file_name = file_name_bytes.decode("utf-8")

print(f"받은 파일명: {file_name}")

# ==========================
# 5. 서버로부터 암호문 수신
# ==========================

# (1) 암호문 길이
encrypted_len_bytes = client.recv(4)
encrypted_len = struct.unpack(">I", encrypted_len_bytes)[0]

# (2) 암호문 본문 수신
encrypted_data = client.recv(encrypted_len)
print(f"암호문 수신 완료 ({encrypted_len} bytes)")

client.close()

# ==========================
# 6. 개인키로 복호화
# ==========================
plaintext = private_key.decrypt(
    encrypted_data,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

print("복호화 완료!")

# ==========================
# 7. 파일명 그대로 저장
# ==========================
with open(file_name, "wb") as f:
    f.write(plaintext)

print(f"복호화된 파일 저장 완료 → {file_name}")
